<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Transformer_Trading_signals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# PART 1 : DATA COLLECTION
# Transformer-Based Adaptive Trading Signals
# ==========================================================

# Install Libraries

!pip install -q pandas numpy matplotlib seaborn scikit-learn ta

# ----------------------------------------------------------
# Imports
# ----------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

# ----------------------------------------------------------
# Upload Dataset
# ----------------------------------------------------------

uploaded = files.upload()

# ----------------------------------------------------------
# Read Dataset
# ----------------------------------------------------------

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

# ----------------------------------------------------------
# Dataset Information
# ----------------------------------------------------------

print("="*60)

print("BTCUSDT LEVEL-2 ORDER BOOK DATASET")

print("="*60)

print("Shape :", df.shape)

print("\nColumns")

print(df.columns.tolist())

print("\nData Types")

print(df.dtypes)

print("\nMissing Values")

print(df.isnull().sum())

print("\nFirst Five Rows")

print(df.head())

  Preparing metadata (setup.py) ... done


Saving BTCUSDT_Depth_1Hour.csv to BTCUSDT_Depth_1Hour.csv
BTCUSDT LEVEL-2 ORDER BOOK DATASET
Shape : (35901, 45)

Columns
['timestamp', 'best_bid', 'best_bid_qty', 'best_ask', 'best_ask_qty', 'bid_price_1', 'bid_qty_1', 'ask_price_1', 'ask_qty_1', 'bid_price_2', 'bid_qty_2', 'ask_price_2', 'ask_qty_2', 'bid_price_3', 'bid_qty_3', 'ask_price_3', 'ask_qty_3', 'bid_price_4', 'bid_qty_4', 'ask_price_4', 'ask_qty_4', 'bid_price_5', 'bid_qty_5', 'ask_price_5', 'ask_qty_5', 'bid_price_6', 'bid_qty_6', 'ask_price_6', 'ask_qty_6', 'bid_price_7', 'bid_qty_7', 'ask_price_7', 'ask_qty_7', 'bid_price_8', 'bid_qty_8', 'ask_price_8', 'ask_qty_8', 'bid_price_9', 'bid_qty_9', 'ask_price_9', 'ask_qty_9', 'bid_price_10', 'bid_qty_10', 'ask_price_10', 'ask_qty_10']

Data Types
timestamp        object
best_bid        float64
best_bid_qty    float64
best_ask        float64
best_ask_qty    float64
bid_price_1     float64
bid_qty_1       float64
ask_price_1     float64
ask_qty_1       float64
bid_price_2     

In [ ]:
# ==========================================================
# COLUMN NAMES
# ==========================================================

print("="*60)

print("COLUMN NAMES")

print("="*60)

for i,col in enumerate(df.columns):

    print(i,"-",col)

COLUMN NAMES
0 - timestamp
1 - best_bid
2 - best_bid_qty
3 - best_ask
4 - best_ask_qty
5 - bid_price_1
6 - bid_qty_1
7 - ask_price_1
8 - ask_qty_1
9 - bid_price_2
10 - bid_qty_2
11 - ask_price_2
12 - ask_qty_2
13 - bid_price_3
14 - bid_qty_3
15 - ask_price_3
16 - ask_qty_3
17 - bid_price_4
18 - bid_qty_4
19 - ask_price_4
20 - ask_qty_4
21 - bid_price_5
22 - bid_qty_5
23 - ask_price_5
24 - ask_qty_5
25 - bid_price_6
26 - bid_qty_6
27 - ask_price_6
28 - ask_qty_6
29 - bid_price_7
30 - bid_qty_7
31 - ask_price_7
32 - ask_qty_7
33 - bid_price_8
34 - bid_qty_8
35 - ask_price_8
36 - ask_qty_8
37 - bid_price_9
38 - bid_qty_9
39 - ask_price_9
40 - ask_qty_9
41 - bid_price_10
42 - bid_qty_10
43 - ask_price_10
44 - ask_qty_10


In [ ]:
# ==========================================================
# PART 2 : FEATURE ENGINEERING
# ==========================================================

import numpy as np

# ----------------------------------------------------------
# Timestamp
# ----------------------------------------------------------

df["timestamp"] = pd.to_datetime(df["timestamp"])

# ----------------------------------------------------------
# Mid Price
# ----------------------------------------------------------

df["MidPrice"] = (

    df["best_bid"]

    +

    df["best_ask"]

) / 2

# ----------------------------------------------------------
# Bid-Ask Spread
# ----------------------------------------------------------

df["Spread"] = (

    df["best_ask"]

    -

    df["best_bid"]

)

# ----------------------------------------------------------
# Mid Price Return
# ----------------------------------------------------------

df["Return"] = (

    df["MidPrice"]

    .pct_change()

)

# ----------------------------------------------------------
# Log Return
# ----------------------------------------------------------

df["Log_Return"] = np.log(

    df["MidPrice"]

    /

    df["MidPrice"].shift(1)

)

# ----------------------------------------------------------
# Order Flow Imbalance (Best Level)
# ----------------------------------------------------------

df["OFI"] = (

    df["best_bid_qty"]

    -

    df["best_ask_qty"]

)

# ----------------------------------------------------------
# Volume Imbalance
# ----------------------------------------------------------

df["Volume_Imbalance"] = (

    (

        df["best_bid_qty"]

        -

        df["best_ask_qty"]

    )

    /

    (

        df["best_bid_qty"]

        +

        df["best_ask_qty"]

    )

)

# ----------------------------------------------------------
# Microprice
# ----------------------------------------------------------

df["Microprice"] = (

    (

        df["best_ask"]

        *

        df["best_bid_qty"]

    )

    +

    (

        df["best_bid"]

        *

        df["best_ask_qty"]

    )

) / (

    df["best_bid_qty"]

    +

    df["best_ask_qty"]

)

# ----------------------------------------------------------
# Depth (Top 10 Levels)
# ----------------------------------------------------------

bid_depth = 0
ask_depth = 0

for i in range(1,11):

    bid_depth += df[f"bid_qty_{i}"]

    ask_depth += df[f"ask_qty_{i}"]

df["BidDepth"] = bid_depth

df["AskDepth"] = ask_depth

# ----------------------------------------------------------
# Depth Imbalance
# ----------------------------------------------------------

df["Depth_Imbalance"] = (

    (

        df["BidDepth"]

        -

        df["AskDepth"]

    )

    /

    (

        df["BidDepth"]

        +

        df["AskDepth"]

    )

)

# ----------------------------------------------------------
# Rolling Volatility
# ----------------------------------------------------------

df["Volatility"] = (

    df["Log_Return"]

    .rolling(20)

    .std()

)

# ----------------------------------------------------------
# Momentum
# ----------------------------------------------------------

df["Momentum"] = (

    df["MidPrice"]

    -

    df["MidPrice"].shift(10)

)

# ----------------------------------------------------------
# Clean Dataset
# ----------------------------------------------------------

df.replace(

    [np.inf,-np.inf],

    np.nan,

    inplace=True

)

df.dropna(inplace=True)

df.reset_index(drop=True,inplace=True)

print("="*60)

print("FEATURE ENGINEERING COMPLETED")

print("="*60)

print(df.head())

print()

print(df.columns.tolist())

FEATURE ENGINEERING COMPLETED
                timestamp  best_bid  best_bid_qty  best_ask  best_ask_qty  \
0 2024-03-01 00:00:02.090  61136.50       1.67475  61136.51       4.54427   
1 2024-03-01 00:00:02.190  61136.50       1.89755  61136.51       4.87317   
2 2024-03-01 00:00:02.290  61136.50       1.80599  61136.51       4.87293   
3 2024-03-01 00:00:03.090  61136.50       1.47882  61136.51       5.56859   
4 2024-03-01 00:00:03.590  61136.31       0.04817  61136.51       4.00397   

   bid_price_1  bid_qty_1  ask_price_1  ask_qty_1  bid_price_2  ...    Return  \
0     61136.50    1.67475     61136.51    4.54427     61136.00  ... -0.000038   
1     61136.50    1.89755     61136.51    4.87317     61134.14  ...  0.000000   
2     61136.50    1.80599     61136.51    4.87293     61136.01  ...  0.000000   
3     61136.50    1.47882     61136.51    5.56859     61134.16  ...  0.000004   
4     61136.31    0.04817     61136.51    4.00397     61134.57  ...  0.000039   

   Log_Return      O

/tmp/ipykernel_1633/1156562042.py:49: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


In [ ]:
# ==========================================================
# PART 3 : BALANCED ADAPTIVE SIGNALS
# ==========================================================

prediction_horizon = 5

df["Future_MidPrice"] = df["MidPrice"].shift(-prediction_horizon)

df["Future_Return"] = (
    df["Future_MidPrice"] - df["MidPrice"]
) / df["MidPrice"]

# ----------------------------------------------------------
# Quantile Thresholds
# ----------------------------------------------------------

upper = df["Future_Return"].quantile(0.70)

lower = df["Future_Return"].quantile(0.30)

print("Upper Threshold :", upper)
print("Lower Threshold :", lower)

# ----------------------------------------------------------
# Labels
# ----------------------------------------------------------

df["Signal"] = 1      # HOLD

df.loc[df["Future_Return"] >= upper, "Signal"] = 2

df.loc[df["Future_Return"] <= lower, "Signal"] = 0

df.dropna(inplace=True)

df.reset_index(drop=True, inplace=True)

print("="*60)
print("SIGNAL DISTRIBUTION")
print("="*60)
print(df["Signal"].value_counts())

Upper Threshold : 5.795592858621106e-05
Lower Threshold : -4.7394191857990345e-05
SIGNAL DISTRIBUTION
Signal
1    1926
2    1448
0    1447
Name: count, dtype: int64


In [ ]:
# ==========================================================
# PART 4 : TRANSFORMER DATASET
# ==========================================================

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ----------------------------------------------------------
# Features
# ----------------------------------------------------------

features = [

    "MidPrice",

    "Spread",

    "Return",

    "Log_Return",

    "OFI",

    "Volume_Imbalance",

    "Microprice",

    "BidDepth",

    "AskDepth",

    "Depth_Imbalance",

    "Volatility",

    "Momentum"

]

X = df[features]

y = df["Signal"]

# ----------------------------------------------------------
# Feature Scaling
# ----------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ----------------------------------------------------------
# Create Sequences
# ----------------------------------------------------------

sequence_length = 30

X_seq = []

y_seq = []

for i in range(sequence_length, len(X_scaled)):

    X_seq.append(

        X_scaled[i-sequence_length:i]

    )

    y_seq.append(

        y.iloc[i]

    )

X_seq = np.array(X_seq)

y_seq = np.array(y_seq)

# ----------------------------------------------------------
# Train-Test Split
# ----------------------------------------------------------

split = int(len(X_seq) * 0.80)

X_train = X_seq[:split]

X_test = X_seq[split:]

y_train = y_seq[:split]

y_test = y_seq[split:]

# ----------------------------------------------------------
# Compute Class Weights
# ----------------------------------------------------------

classes = np.unique(y_train)

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=y_train

)

class_weight = dict(zip(classes, weights))

print("="*60)

print("CLASS WEIGHTS")

print("="*60)

print(class_weight)

# ----------------------------------------------------------
# Dataset Information
# ----------------------------------------------------------

print("="*60)

print("TRANSFORMER DATASET")

print("="*60)

print("Sequence Shape :", X_seq.shape)

print("Training Shape :", X_train.shape)

print("Testing Shape  :", X_test.shape)

print("Target Shape   :", y_seq.shape)

print("Features       :", X_train.shape[2])

print("Classes        :", np.unique(y_train))

CLASS WEIGHTS
{np.int64(0): np.float64(1.1116913257905425), np.int64(1): np.float64(0.8425681618293756), np.int64(2): np.float64(1.0945444158811768)}
TRANSFORMER DATASET
Sequence Shape : (4791, 30, 12)
Training Shape : (3832, 30, 12)
Testing Shape  : (959, 30, 12)
Target Shape   : (4791,)
Features       : 12
Classes        : [0 1 2]


In [ ]:
# ==========================================================
# CLASS DISTRIBUTION
# ==========================================================

import pandas as pd

signal_names = {

    0:"SELL",

    1:"HOLD",

    2:"BUY"

}

distribution = pd.Series(y_train).map(signal_names).value_counts()

print("="*60)

print("TRAINING SIGNAL DISTRIBUTION")

print("="*60)

print(distribution)

TRAINING SIGNAL DISTRIBUTION
HOLD    3319
BUY      283
SELL     234
Name: count, dtype: int64


In [ ]:
# ==========================================================
# PART 5 : TRANSFORMER CLASSIFIER
# ==========================================================

import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model

# ----------------------------------------------------------
# Input Layer
# ----------------------------------------------------------

inputs = Input(
    shape=(X_train.shape[1], X_train.shape[2])
)

# ----------------------------------------------------------
# Transformer Encoder Block 1
# ----------------------------------------------------------

attention = MultiHeadAttention(
    num_heads=4,
    key_dim=16
)(inputs, inputs)

x = LayerNormalization()(inputs + attention)

ffn = Dense(
    128,
    activation="relu"
)(x)

ffn = Dense(
    X_train.shape[2]
)(ffn)

x = LayerNormalization()(x + ffn)

# ----------------------------------------------------------
# Transformer Encoder Block 2
# ----------------------------------------------------------

attention = MultiHeadAttention(
    num_heads=4,
    key_dim=16
)(x, x)

x2 = LayerNormalization()(x + attention)

ffn = Dense(
    128,
    activation="relu"
)(x2)

ffn = Dense(
    X_train.shape[2]
)(ffn)

x = LayerNormalization()(x2 + ffn)

# ----------------------------------------------------------
# Pooling
# ----------------------------------------------------------

x = GlobalAveragePooling1D()(x)

# ----------------------------------------------------------
# Dense Layers
# ----------------------------------------------------------

x = Dense(
    64,
    activation="relu"
)(x)

x = Dropout(0.30)(x)

x = Dense(
    32,
    activation="relu"
)(x)

# ----------------------------------------------------------
# Output Layer
# ----------------------------------------------------------

outputs = Dense(
    3,
    activation="softmax"
)(x)

# ----------------------------------------------------------
# Build Model
# ----------------------------------------------------------

model = Model(
    inputs,
    outputs
)

# ----------------------------------------------------------
# Compile
# ----------------------------------------------------------

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 30, 12)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 12)    │      3,276 │ input_layer[0][0… │
│ (MultiHeadAttentio… │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 30, 12)    │          0 │ input_layer[0][0… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 30, 12)    │         24 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30, 128)   │      1,664 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 30, 12)    │      1,548 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 30, 12)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 12)    │         24 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 12)    │      3,276 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 30, 12)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 12)    │         24 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 30, 128)   │      1,664 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 30, 12)    │      1,548 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 30, 12)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 12)    │         24 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 12)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │        832 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 32)        │      2,080 │ dropout_2[0][0] 

 Total params: 16,083 (62.82 KB)

 Trainable params: 16,083 (62.82 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ==========================================================
# PART 6 : OPTUNA + FINAL TRAINING
# ==========================================================

!pip install -q optuna

import optuna
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import *

# ----------------------------------------------------------
# Optuna Objective
# ----------------------------------------------------------

def objective(trial):

    heads = trial.suggest_categorical(
        "heads",[2,4,8]
    )

    key_dim = trial.suggest_categorical(
        "key_dim",[8,16,32]
    )

    ff_units = trial.suggest_categorical(
        "ff_units",[64,128,256]
    )

    dropout = trial.suggest_float(
        "dropout",0.2,0.5
    )

    lr = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True
    )

    batch = trial.suggest_categorical(
        "batch_size",
        [32,64,128]
    )

    inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))

    att = MultiHeadAttention(
        num_heads=heads,
        key_dim=key_dim
    )(inputs, inputs)

    x = LayerNormalization()(inputs + att)

    ffn = Dense(
        ff_units,
        activation="relu"
    )(x)

    ffn = Dense(X_train.shape[2])(ffn)

    x = LayerNormalization()(x + ffn)

    x = GlobalAveragePooling1D()(x)

    x = Dropout(dropout)(x)

    x = Dense(
        64,
        activation="relu"
    )(x)

    outputs = Dense(
        3,
        activation="softmax"
    )(x)

    model = Model(inputs, outputs)

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=lr
        ),

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    history = model.fit(

        X_train,

        y_train,

        validation_split=0.20,

        epochs=10,

        batch_size=batch,

        class_weight=class_weight,

        verbose=0

    )

    return max(history.history["val_accuracy"])

# ----------------------------------------------------------
# Run Optuna
# ----------------------------------------------------------

study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=10
)

print("="*60)
print("BEST PARAMETERS")
print("="*60)
print(study.best_params)

params = study.best_params

# ----------------------------------------------------------
# Final Transformer
# ----------------------------------------------------------

inputs = Input(
    shape=(X_train.shape[1], X_train.shape[2])
)

att = MultiHeadAttention(

    num_heads=params["heads"],

    key_dim=params["key_dim"]

)(inputs, inputs)

x = LayerNormalization()(inputs + att)

ffn = Dense(

    params["ff_units"],

    activation="relu"

)(x)

ffn = Dense(
    X_train.shape[2]
)(ffn)

x = LayerNormalization()(x + ffn)

x = GlobalAveragePooling1D()(x)

x = Dropout(
    params["dropout"]
)(x)

x = Dense(
    64,
    activation="relu"
)(x)

outputs = Dense(
    3,
    activation="softmax"
)(x)

model = Model(inputs, outputs)

model.compile(

    optimizer=tf.keras.optimizers.Adam(

        learning_rate=params["learning_rate"]

    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

callbacks = [

    EarlyStopping(

        monitor="val_loss",

        patience=5,

        restore_best_weights=True

    ),

    ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=3,

        min_lr=1e-6

    )

]

# ----------------------------------------------------------
# Final Training
# ----------------------------------------------------------

history = model.fit(

    X_train,

    y_train,

    validation_split=0.20,

    epochs=40,

    batch_size=params["batch_size"],

    callbacks=callbacks,

    class_weight=class_weight,

    verbose=1

)

# ----------------------------------------------------------
# Prediction
# ----------------------------------------------------------

pred_prob = model.predict(X_test)

pred = np.argmax(
    pred_prob,
    axis=1
)

print("="*60)
print("TRAINING COMPLETED")
print("="*60)

[I 2026-08-04 02:49:33,850] A new study created in memory with name: no-name-848b5783-42cc-4ac1-a8b4-7de5548f2867
[I 2026-08-04 02:49:47,442] Trial 0 finished with value: 0.38591915369033813 and parameters: {'heads': 8, 'key_dim': 8, 'ff_units': 256, 'dropout': 0.25880705974223406, 'learning_rate': 0.006779898934599998, 'batch_size': 64}. Best is trial 0 with value: 0.38591915369033813.
[I 2026-08-04 02:50:00,703] Trial 1 finished with value: 0.37418514490127563 and parameters: {'heads': 2, 'key_dim': 32, 'ff_units': 256, 'dropout': 0.4946856129381073, 'learning_rate': 0.0025568154942416407, 'batch_size': 128}. Best is trial 0 with value: 0.38591915369033813.
[I 2026-08-04 02:50:14,792] Trial 2 finished with value: 0.4198174774646759 and parameters: {'heads': 2, 'key_dim': 8, 'ff_units': 256, 'dropout': 0.377807562155187, 'learning_rate': 0.0001727227995906631, 'batch_size': 64}. Best is trial 2 with value: 0.4198174774646759.
[I 2026-08-04 02:50:33,106] Trial 3 finished with value: 0.

BEST PARAMETERS
{'heads': 2, 'key_dim': 16, 'ff_units': 64, 'dropout': 0.38873777684279914, 'learning_rate': 0.00029327210220962994, 'batch_size': 128}
Epoch 1/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 199ms/step - accuracy: 0.3416 - loss: 1.1691 - val_accuracy: 0.3220 - val_loss: 1.1237 - learning_rate: 2.9327e-04
Epoch 2/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3488 - loss: 1.1184 - val_accuracy: 0.3611 - val_loss: 1.1065 - learning_rate: 2.9327e-04
Epoch 3/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3595 - loss: 1.0977 - val_accuracy: 0.3690 - val_loss: 1.0972 - learning_rate: 2.9327e-04
Epoch 4/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3677 - loss: 1.0884 - val_accuracy: 0.3872 - val_loss: 1.0919 - learning_rate: 2.9327e-04
Epoch 5/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3759 - loss: 1.0891 - val_accuracy: 0.3611 - val_loss: 1.0925 - learning_rate: 2.9327e-04
Epoch 6/40
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3674 - loss

In [ ]:

# ==========================================================
# PART 7 : MODEL EVALUATION
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_test,pred)

precision = precision_score(
    y_test,
    pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    pred,
    average="weighted",
    zero_division=0
)

print("="*60)
print("TRANSFORMER RESULTS")
print("="*60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report\n")

print(

    classification_report(

        y_test,

        pred,

        target_names=["SELL","HOLD","BUY"],

        zero_division=0

    )

)

print("\nConfusion Matrix\n")

cm = confusion_matrix(

    y_test,

    pred

)

print(cm)

TRANSFORMER RESULTS
Accuracy : 0.3650
Precision: 0.3682
Recall   : 0.3650
F1 Score : 0.3343

Classification Report

              precision    recall  f1-score   support

        SELL       0.36      0.70      0.47       293
        HOLD       0.39      0.25      0.30       390
         BUY       0.35      0.17      0.23       276

    accuracy                           0.36       959
   macro avg       0.37      0.37      0.34       959
weighted avg       0.37      0.36      0.33       959


Confusion Matrix

[[206  67  20]
 [226  97  67]
 [143  86  47]]


In [ ]:
# ==========================================================
# PREDICTION CONFIDENCE
# ==========================================================

confidence = pred_prob.max(axis=1)

print("="*60)

print("PREDICTION CONFIDENCE")

print("="*60)

print(pd.Series(confidence).describe())

PREDICTION CONFIDENCE
count    959.000000
mean       0.376217
std        0.017266
min        0.337219
25%        0.362413
50%        0.374949
75%        0.390975
max        0.417911
dtype: float64


In [ ]:
# ==========================================================
# SIGNAL DISTRIBUTION
# ==========================================================

signal_names = {

    0:"SELL",

    1:"HOLD",

    2:"BUY"

}

actual = pd.Series(y_test).map(signal_names)

predicted = pd.Series(pred).map(signal_names)

print("="*60)

print("ACTUAL SIGNALS")

print("="*60)

print(actual.value_counts())

print()

print("="*60)

print("PREDICTED SIGNALS")

print("="*60)

print(predicted.value_counts())

ACTUAL SIGNALS
HOLD    390
SELL    293
BUY     276
Name: count, dtype: int64

PREDICTED SIGNALS
SELL    575
HOLD    250
BUY     134
Name: count, dtype: int64
